In [ ]:
import pandas as pd 
import sqlite3
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
con = sqlite3.connect(r'C:\Users\thway\ipl_cricket-analatics\notebooks\ipl.db') 
  
def q(sql): 
    """Run a SELECT and show the result as a table.""" 
    return pd.read_sql(sql, con)


def run_sql_file(path): 
    """Read a .sql file and execute every statement in it.""" 
    with open(path, 'r') as f: 
        script = f.read() 
    con.executescript(script) 
    con.commit() 
    print('ran', path)

In [3]:
stats.binomtest(613, 1187, 0.5).pvalue 
# 0.2700  <- the toss result from 4.2, in one line

np.float64(0.2700389712415702)

In [4]:
# Quick check: SciPy is working
check_p = stats.binomtest(613, 1187, 0.5).pvalue
print("Check p-value:", check_p)


Check p-value: 0.2700389712415702


# T1 — Does winning the toss help you win the match?
Finding: Toss winners won 51.64% of decisive matches.
Statistical question: Is the toss winner's match-win rate significantly different from 50%?
Null hypothesis: H₀: Toss winners win 50% of decisive matches.
Test: Binomial test

In [5]:
# T1 — SQL
t1_df = q("""
SELECT
    COUNT(*) AS n,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS k
FROM matches_clean
WHERE result = 'win';
""")

display(t1_df)

,n,k
0,1187,613


In [6]:
# T1 — Statistical test
n = int(t1_df.loc[0, "n"])
k = int(t1_df.loc[0, "k"])

observed_rate = k / n
null_rate = 0.50

result = stats.binomtest(k, n, null_rate)
p_value = result.pvalue

ci_low, ci_high = proportion_confint(
    k, n, method="wilson"
)

effect_pp = (observed_rate - null_rate) * 100

print(f"Sample size: {n}")
print(f"Toss-winner wins: {k}")
print(f"Observed rate: {observed_rate*100:.2f}%")
print(f"p-value: {p_value:.6f}")
print(f"Effect: {effect_pp:.2f} percentage points")
print(f"95% CI: {ci_low*100:.2f}% to {ci_high*100:.2f}%")

if p_value < 0.05:
    print("Verdict: Statistically significant evidence of a difference from 50%.")
else:
    print("Verdict: Not enough evidence to conclude that the toss-winner win rate differs from 50%.")

Sample size: 1187
Toss-winner wins: 613
Observed rate: 51.64%
p-value: 0.270039
Effect: 1.64 percentage points
95% CI: 48.80% to 54.48%
Verdict: Not enough evidence to conclude that the toss-winner win rate differs from 50%.


# T2 — Does the decision taken after the toss matter?
Finding: Toss winners who chose to field had a different win rate from those who chose to bat.
Statistical question: Is toss decision associated with whether the toss winner wins the match?
Null hypothesis: H₀: Toss decision and toss-winner match outcome are independent.
Test: Chi-square test

In [7]:
# T2 — SQL
t2_df = q("""
SELECT
    toss_decision,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS won,
    SUM(
        CASE
            WHEN toss_winner <> match_winner THEN 1
            ELSE 0
        END
    ) AS lost
FROM matches_clean
WHERE result = 'win'
GROUP BY toss_decision;
""")

display(t2_df)

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [8]:
# T2 — Chi-square + Cramer's V
table = t2_df[["won", "lost"]].to_numpy()

chi2, p_value, dof, expected = stats.chi2_contingency(table)

n = table.sum()
cramers_v = np.sqrt(chi2 / n)

print(f"Chi-square: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")
print(f"Sample size: {n}")
print(f"Cramer's V: {cramers_v:.4f}")

if p_value < 0.05:
    print("Verdict: Evidence of an association between toss decision and toss-winner outcome.")
else:
    print("Verdict: Not enough evidence of an association between toss decision and toss-winner outcome.")

Chi-square: 8.0401
Degrees of freedom: 1
p-value: 0.004575
Sample size: 1187
Cramer's V: 0.0823
Verdict: Evidence of an association between toss decision and toss-winner outcome.


# T3 — Does the chasing team really have an advantage?

**Finding:** Chasing teams won 54.51% of decisive matches.

**Statistical question:** Is the chasing win rate significantly different from 50%?

**Null hypothesis:** H₀: Chasing teams win 50% of matches.

**Test:** Binomial test


In [9]:
# T3 — SQL
t3_df = q("""
SELECT
    COUNT(*) AS n,
    SUM(chase_won) AS k
FROM v_match_totals;
""")

display(t3_df)

,n,k
0,1187,647


In [10]:
# T3 — Statistical test
n = int(t3_df.loc[0, "n"])
k = int(t3_df.loc[0, "k"])

observed_rate = k / n
null_rate = 0.50

result = stats.binomtest(k, n, null_rate)
p_value = result.pvalue

ci_low, ci_high = proportion_confint(
    k, n, method="wilson"
)

effect_pp = (observed_rate - null_rate) * 100

print(f"Sample size: {n}")
print(f"Chases won: {k}")
print(f"Observed chase win rate: {observed_rate*100:.2f}%")
print(f"p-value: {p_value:.6f}")
print(f"Effect: {effect_pp:.2f} percentage points")
print(f"95% CI: {ci_low*100:.2f}% to {ci_high*100:.2f}%")

if p_value < 0.05:
    print("Verdict: Evidence that the chase win rate differs from 50%.")
else:
    print("Verdict: Not enough evidence to conclude that the chase win rate differs from 50%.")

Sample size: 1187
Chases won: 647
Observed chase win rate: 54.51%
p-value: 0.002080
Effect: 4.51 percentage points
95% CI: 51.66% to 57.32%
Verdict: Evidence that the chase win rate differs from 50%.


# T4 — Does the chase win rate fall as the target rises?

**Finding:** Chase success changes across target bands.

**Statistical question:** Is chase outcome associated with the target band?

**Null hypothesis:** H₀: Chase outcome is independent of target band.

**Test:** Chi-square test

In [11]:
# T4 — SQL
t4_df = q("""
SELECT
    CASE
        WHEN first_innings_runs < 140 THEN 'Under 140'
        WHEN first_innings_runs < 160 THEN '140-159'
        WHEN first_innings_runs < 180 THEN '160-179'
        WHEN first_innings_runs < 200 THEN '180-199'
        ELSE '200+'
    END AS band,
    SUM(chase_won) AS won,
    SUM(1 - chase_won) AS lost
FROM v_match_totals
GROUP BY band;
""")

display(t4_df)

,band,won,lost
0,140-159,171,70
1,160-179,166,146
2,180-199,83,134
3,200+,46,152
4,Under 140,181,38


In [12]:
# T4 — Chi-square + Cramer's V
table = t4_df[["won", "lost"]].to_numpy()

chi2, p_value, dof, expected = stats.chi2_contingency(table)

n = table.sum()
cramers_v = np.sqrt(chi2 / n)

t4_display = t4_df.copy()
t4_display["chase_win_rate"] = (
    t4_display["won"] /
    (t4_display["won"] + t4_display["lost"]) * 100
)

display(t4_display)

print(f"Chi-square: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6e}")
print(f"Sample size: {n}")
print(f"Cramer's V: {cramers_v:.4f}")

if p_value < 0.05:
    print("Verdict: Evidence that chase outcome differs across target bands.")
else:
    print("Verdict: Not enough evidence of a difference across target bands.")

,band,won,lost,chase_win_rate
0,140-159,171,70,70.954357
1,160-179,166,146,53.205128
2,180-199,83,134,38.248848
3,200+,46,152,23.232323
4,Under 140,181,38,82.648402


Chi-square: 197.6789
Degrees of freedom: 4
p-value: 1.185422e-41
Sample size: 1187
Cramer's V: 0.4081
Verdict: Evidence that chase outcome differs across target bands.


# T5 — Are defended totals actually bigger than chased totals?

**Finding:** Matches in which the target was successfully defended had higher first-innings totals than successful chases.

**Statistical question:** Do defended and successfully chased matches have different average first-innings totals?

**Null hypothesis:** H₀: The two groups have the same mean first-innings total.

**Test:** Welch independent t-test

**Effect size:** Cohen's d

In [13]:
# T5 — SQL
t5_df = q("""
SELECT
    first_innings_runs,
    chase_won
FROM v_match_totals;
""")

display(t5_df.head())

,first_innings_runs,chase_won
0,222,0
1,207,0
2,184,1
3,183,1
4,163,1


In [14]:
# T5 — Independent t-test + Cohen's d
defended = t5_df.loc[
    t5_df["chase_won"] == 0, "first_innings_runs"
].dropna()

chased = t5_df.loc[
    t5_df["chase_won"] == 1, "first_innings_runs"
].dropna()

t_stat, p_value = stats.ttest_ind(
    defended,
    chased,
    equal_var=False
)

pooled_sd = np.sqrt(
    (defended.std(ddof=1)**2 + chased.std(ddof=1)**2) / 2
)

cohens_d = (
    defended.mean() - chased.mean()
) / pooled_sd

print(f"Defended n: {len(defended)}")
print(f"Chased n: {len(chased)}")
print(f"Defended mean: {defended.mean():.2f}")
print(f"Chased mean: {chased.mean():.2f}")
print(f"Mean difference: {defended.mean() - chased.mean():.2f} runs")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6e}")
print(f"Cohen's d: {cohens_d:.3f}")

if p_value < 0.05:
    print("Verdict: Evidence that the two groups have different mean first-innings totals.")
else:
    print("Verdict: Not enough evidence of a difference in mean first-innings totals.")

Defended n: 540
Chased n: 647
Defended mean: 183.34
Chased mean: 155.54
Mean difference: 27.80 runs
t-statistic: 15.7302
p-value: 1.172723e-50
Cohen's d: 0.916
Verdict: Evidence that the two groups have different mean first-innings totals.


# T6 — Is the Powerplay really faster than the Middle overs?

**Finding:** Powerplay run rate appeared higher than Middle-over run rate.

**Statistical question:** Within the same innings, is Powerplay run rate different from Middle-over run rate?

**Null hypothesis:** H₀: The mean Powerplay and Middle-over run rates are equal.

**Test:** Paired t-test

The observations are paired because both rates come from the same innings.

In [15]:
# T6 — SQL
t6_df = q("""
SELECT
    match_id,
    innings,
    6.0 * SUM(
        CASE
            WHEN phase = 'Powerplay' THEN total_runs
            ELSE 0
        END
    )
    / SUM(
        CASE
            WHEN phase = 'Powerplay' THEN is_legal
            ELSE 0
        END
    ) AS pp,
    6.0 * SUM(
        CASE
            WHEN phase = 'Middle' THEN total_runs
            ELSE 0
        END
    )
    / SUM(
        CASE
            WHEN phase = 'Middle' THEN is_legal
            ELSE 0
        END
    ) AS mid
FROM v_ball
WHERE innings IN (1, 2)
GROUP BY match_id, innings;
""")

display(t6_df.head())

,match_id,innings,pp,mid
0,335982,1,10.166667,10.333333
1,335982,2,4.333333,6.111111
2,335983,1,8.833333,12.000000
3,335983,2,10.500000,11.333333
4,335984,1,6.666667,6.222222


In [16]:
# T6 — Paired t-test + paired Cohen's d
t6_clean = t6_df[["pp", "mid"]].dropna()

t_stat, p_value = stats.ttest_rel(
    t6_clean["pp"],
    t6_clean["mid"]
)

differences = t6_clean["pp"] - t6_clean["mid"]
paired_d = differences.mean() / differences.std(ddof=1)

print(f"Innings measured: {len(t6_clean)}")
print(f"Powerplay mean: {t6_clean['pp'].mean():.2f}")
print(f"Middle-over mean: {t6_clean['mid'].mean():.2f}")
print(f"Mean difference: {differences.mean():.2f} runs/over")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Paired Cohen's d: {paired_d:.3f}")

if p_value < 0.05:
    print("Verdict: Evidence of a difference between Powerplay and Middle-over run rates.")
else:
    print("Verdict: Not enough evidence to conclude that the two run rates differ.")

Innings measured: 2406
Powerplay mean: 8.02
Middle-over mean: 7.95
Mean difference: 0.08 runs/over
t-statistic: 1.5311
p-value: 0.125866
Paired Cohen's d: 0.031
Verdict: Not enough evidence to conclude that the two run rates differ.


# T7 — Do left-handers really hit harder than right-handers?

**Finding:** Left- and right-handed batters may have different strike rates.

**Statistical question:** Do left-handed and right-handed batters have different average strike rates?

**Null hypothesis:** H₀: The two groups have the same mean batter strike rate.

**Test:** Welch independent t-test

Only batters with at least 500 legal balls are included, with one strike rate per batter.

In [17]:
# T7 — SQL
t7_df = q("""
SELECT
    batter,
    batsman_type,
    100.0 * SUM(batter_runs) / SUM(is_legal) AS sr,
    SUM(is_legal) AS balls
FROM v_ball
WHERE batsman_type IS NOT NULL
GROUP BY batter, batsman_type
HAVING balls >= 500;
""")

display(t7_df.head())

,batter,batsman_type,sr,balls
0,A Badoni,Right hand Bat,139.263804,815
1,A Symonds,Right hand Bat,130.738255,745
2,AB de Villiers,Right hand Bat,152.181604,3392
3,AC Gilchrist,Left hand Bat,139.514498,1483
4,AD Mathews,Right hand Bat,126.352531,573


In [18]:
# T7 — Independent t-test + Cohen's d
left = t7_df.loc[
    t7_df["batsman_type"] == "Left hand Bat", "sr"
].dropna()

right = t7_df.loc[
    t7_df["batsman_type"] == "Right hand Bat", "sr"
].dropna()

t_stat, p_value = stats.ttest_ind(
    left,
    right,
    equal_var=False
)

pooled_sd = np.sqrt(
    (left.std(ddof=1)**2 + right.std(ddof=1)**2) / 2
)

cohens_d = (
    left.mean() - right.mean()
) / pooled_sd

print(f"Left-handed batters: {len(left)}")
print(f"Right-handed batters: {len(right)}")
print(f"Left average SR: {left.mean():.2f}")
print(f"Right average SR: {right.mean():.2f}")
print(f"Mean difference: {left.mean() - right.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Cohen's d: {cohens_d:.3f}")

if p_value < 0.05:
    print("Verdict: Evidence of a difference in average strike rate between the two groups.")
else:
    print("Verdict: Not enough evidence to conclude that the two groups differ in average strike rate.")

Left-handed batters: 50
Right-handed batters: 87
Left average SR: 136.60
Right average SR: 136.62
Mean difference: -0.02
t-statistic: -0.0073
p-value: 0.994161
Cohen's d: -0.001
Verdict: Not enough evidence to conclude that the two groups differ in average strike rate.


# T8 — Are spinners genuinely cheaper than fast bowlers?

**Finding:** Spin bowlers may have a lower economy rate than pace bowlers.

**Statistical question:** Do spin and pace bowlers have different average economy rates?

**Null hypothesis:** H₀: Spin and pace bowlers have the same mean economy rate.

**Test:** Welch independent t-test

Only bowlers with at least 500 legal balls are included.

In [19]:
# T8 — SQL
t8_df = q("""
SELECT
    bowler,
    CASE
        WHEN bowler_type LIKE '%Legbreak%'
          OR bowler_type LIKE '%Offbreak%'
          OR bowler_type LIKE '%Orthodox%'
          OR bowler_type LIKE '%Googly%'
          OR bowler_type LIKE '%Wrist spin%'
        THEN 'Spin'
        ELSE 'Pace'
    END AS style,
    6.0 * SUM(bowler_runs) / SUM(is_legal) AS economy,
    SUM(is_legal) AS balls
FROM v_ball
WHERE TRIM(COALESCE(bowler_type, '')) <> ''
GROUP BY bowler, style
HAVING balls >= 500;
""")

display(t8_df.head())

,bowler,style,economy,balls
0,A Kumble,Spin,6.578238,965
1,A Mishra,Spin,7.377633,3371
2,A Nehra,Pace,7.845912,1908
3,A Nortje,Pace,9.085714,1120
4,A Symonds,Spin,7.673624,527


In [20]:
# T8 — Independent t-test + Cohen's d
spin = t8_df.loc[
    t8_df["style"] == "Spin", "economy"
].dropna()

pace = t8_df.loc[
    t8_df["style"] == "Pace", "economy"
].dropna()

t_stat, p_value = stats.ttest_ind(
    spin,
    pace,
    equal_var=False
)

pooled_sd = np.sqrt(
    (spin.std(ddof=1)**2 + pace.std(ddof=1)**2) / 2
)

cohens_d = (
    spin.mean() - pace.mean()
) / pooled_sd

print(f"Spin bowlers: {len(spin)}")
print(f"Pace bowlers: {len(pace)}")
print(f"Spin average economy: {spin.mean():.2f}")
print(f"Pace average economy: {pace.mean():.2f}")
print(f"Mean difference: {spin.mean() - pace.mean():.2f} runs/over")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6e}")
print(f"Cohen's d: {cohens_d:.3f}")

if p_value < 0.05:
    print("Verdict: Evidence that the two bowling styles have different average economy rates.")
else:
    print("Verdict: Not enough evidence to conclude that their average economy rates differ.")

Spin bowlers: 51
Pace bowlers: 98
Spin average economy: 7.70
Pace average economy: 8.49
Mean difference: -0.79 runs/over
t-statistic: -7.0756
p-value: 1.124965e-10
Cohen's d: -1.187
Verdict: Evidence that the two bowling styles have different average economy rates.


# T9 — Does Sawai Mansingh favour the chasing team?

**Finding:** Sawai Mansingh Stadium had a 65.15% chase win rate.

**Statistical question:** Is the venue's chase win rate different from the league-wide chase rate?

**Null hypothesis:** H₀: The venue's chase success rate is the league-wide rate of 54.51%.

**Test:** Binomial test

**Important:** Compare the venue to **54.51%**, not to 50%.

In [21]:
# T9 — SQL
# Both venue names represent Sawai Mansingh Stadium in this dataset

t9_df = q("""
SELECT
    COUNT(*) AS n,
    SUM(chase_won) AS k
FROM v_match_totals
WHERE venue IN (
    'Sawai Mansingh Stadium',
    'Sawai Mansingh Stadium, Jaipur'
);
""")

display(t9_df)

,n,k
0,66,43


In [22]:
# T9 — Binomial test against league-wide rate
n = int(t9_df.loc[0, "n"])
k = int(t9_df.loc[0, "k"])

league_rate = 0.5451
venue_rate = k / n

result = stats.binomtest(
    k,
    n,
    league_rate
)

p_value = result.pvalue

ci_low, ci_high = proportion_confint(
    k, n, method="wilson"
)

effect_pp = (venue_rate - league_rate) * 100

print(f"Matches at venue: {n}")
print(f"Chases won: {k}")
print(f"Venue chase rate: {venue_rate*100:.2f}%")
print(f"League chase rate: {league_rate*100:.2f}%")
print(f"Effect: {effect_pp:.2f} percentage points")
print(f"p-value: {p_value:.6f}")
print(f"95% CI: {ci_low*100:.2f}% to {ci_high*100:.2f}%")

if p_value < 0.05:
    print("Verdict: Evidence that the venue's chase rate differs from the league rate.")
else:
    print("Verdict: Not enough evidence to conclude that the venue's chase rate differs from the league rate.")

Matches at venue: 66
Chases won: 43
Venue chase rate: 65.15%
League chase rate: 54.51%
Effect: 10.64 percentage points
p-value: 0.084934
95% CI: 53.11% to 75.52%
Verdict: Not enough evidence to conclude that the venue's chase rate differs from the league rate.


# T10 — Has scoring genuinely risen across the seasons?

**Finding:** Average innings score appears to have increased over the seasons.

**Statistical question:** Is season year positively associated with average innings score?

**Null hypothesis:** H₀: There is no linear association between season year and average innings score.

**Test:** Pearson correlation

**Required note:** Correlation does not establish causation, so the report must name at least one alternative explanation.

In [23]:
# T10 — SQL: one row per season
t10_df = q("""
SELECT
    m.season_year,
    AVG(i.runs) AS avg_score
FROM matches_clean m
JOIN v_innings i
    ON i.match_id = m.match_id
GROUP BY m.season_year
ORDER BY m.season_year;
""")

display(t10_df)

,season_year,avg_score
0,2008,154.629310
1,2009,143.157895
2,2010,157.200000
3,2011,145.889655
4,2012,151.709459
5,2013,148.296053
6,2014,157.575000
7,2015,156.683761
8,2016,157.183333
9,2017,159.059322


In [24]:
# T10 — Pearson correlation
# Convert season_year to a numeric series and handle the 2020/21 season label
season_year_numeric = pd.to_numeric(
    t10_df["season_year"].replace("2020/21", "2020.5"),
    errors="coerce"
)
avg_score_numeric = pd.to_numeric(t10_df["avg_score"], errors="coerce")

valid = ~season_year_numeric.isna() & ~avg_score_numeric.isna()

r, p_value = stats.pearsonr(
    season_year_numeric[valid],
    avg_score_numeric[valid]
)

r_squared = r ** 2

print(f"Seasons: {len(t10_df)}")
print(f"Pearson r: {r:.3f}")
print(f"R-squared: {r_squared:.3f}")
print(f"p-value: {p_value:.10f}")

if p_value < 0.05:
    print("Verdict: Evidence of a linear association between season year and average innings score.")
else:
    print("Verdict: Not enough evidence of a linear association.")

print(
    "Alternative explanation: changes in bat technology, rules, boundary sizes, "
    "venues/pitches, or team composition could also contribute to the trend."
)

Seasons: 19
Pearson r: 0.860
R-squared: 0.739
p-value: 0.0000023908
Verdict: Evidence of a linear association between season year and average innings score.
Alternative explanation: changes in bat technology, rules, boundary sizes, venues/pitches, or team composition could also contribute to the trend.


# Step 4.15 — Test Three of My Own Findings

Choose **three findings from your Step 3 Analysis Report** where you have the least confidence.

For each one, write the claim, statistical question, null hypothesis, and test before adding the code.

## Finding 1

**Finding:**  
TODO — paste your Step 3 finding here.

**Statistical question:**  
TODO

**Null hypothesis:**  
H₀: TODO

**Test selected:**  
TODO

## Finding 2

**Finding:**  
TODO — paste your Step 3 finding here.

**Statistical question:**  
TODO

**Null hypothesis:**  
H₀: TODO

**Test selected:**  
TODO

## Finding 3

**Finding:**  
TODO — paste your Step 3 finding here.

**Statistical question:**  
TODO

**Null hypothesis:**  
H₀: TODO

**Test selected:**  
TODO